# Uniprot Usage Demo
This document demonstrates how to use the Uniprot API implementation programmatically.

This api permits searching via 3 ways:
- **Stream search**: This method is suitable for text queries. It uses the same search syntax as the Uniprot website.
- **ID search**: This method is suitable for searching specific entries by their Uniprot IDs for a given pandas DataFrame.
- **Sequence search**: This method is suitable for searching entries by their protein sequences. It uses the BLAST algorithm to find similar sequences.


In [13]:
from bioseq_dl import UniprotInterface
import pandas as pd

## Stream Search

### Define the query
- Query: Should be defined as a string containing the search criteria. The query syntax is the same as the one used in the Uniprot website.
- Fields: List of entry sections to be returned. More fields can be found in the [Uniprot documentation](https://rest.uniprot.org/configure/uniprotkb/result-fields).
- Sort: Specify field by wich to sort results.

In [36]:
query="organism_name:homo sapiens (human) AND length:[15 TO 30] AND reviewed:true"
fields="accession,protein_name,sequence,ec,lineage,organism_name,xref_pfam,xref_alphafolddb,xref_pdb,go_id"
sort="accession asc"

### Instantiating the API

In [37]:
instance = UniprotInterface(
    total_retries=5
)

### Making the request

In [38]:
response = instance.submit_stream(
    query=query,
    fields=fields,
    sort=sort,
    include_isoform=True,
    download=False,
    format="json"
)

### Parsing results

In [41]:
instance.parse_stream_response(
    query=query,
    response=response,
    extract_fields=None
).head(5)

,query,accession,protein_name,organism_name,taxon_id,ineage,sequence,length,alphafold_ids,biogrid_ids,...,interpro_ids,kegg_ids,panther_ids,pathwaycommons_ids,pdb_ids,pfam_ids,pride_ids,reactome_ids,refseq_ids,string_ids
0,organism_name:homo sapiens (human) AND length:...,A0A075B6S0,T cell receptor gamma joining 1,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",NYYKKLFGSGTTLVVT,16,[A0A075B6S0],[],...,[],[],[],[],[],[],[],[],[],[]
1,organism_name:homo sapiens (human) AND length:...,A0A075B6Y3,T cell receptor alpha joining 3,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",GYSSASKIIFGSGTRLSIRP,20,[A0A075B6Y3],[],...,[],[],[],[],[],[],[],[],[],[]
2,organism_name:homo sapiens (human) AND length:...,A0A075B6Y9,T cell receptor alpha joining 42,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",YGGSQGNLIFGKGTKLSVKP,20,[A0A075B6Y9],[],...,[],[],[],[],[],[],[],[],[],[]
3,organism_name:homo sapiens (human) AND length:...,A0A075B700,T cell receptor alpha joining 31,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",NNNARLMFGDGTQLVVKP,18,[A0A075B700],[],...,[],[],[],[],[],[],[],[],[],[]
4,organism_name:homo sapiens (human) AND length:...,A0A075B706,T cell receptor delta joining 1,Homo sapiens,9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",TDKLIFGKGTRVTVEP,16,[A0A075B706],[],...,[],[],[],[],[],[],[],[],[],[]


## ID Search

### Define the query
A query has to be defined. This query should contain:
- Identifiers: A list of Uniprot IDs to be fetched.
- From db: The database from which the IDs originate. In this case, it is "UniProtKB_AC-ID".
- To db: The target database to which the IDs will be mapped. In this case,
it is "UniProtKB".

In [22]:
df = pd.DataFrame({
    "ids": ["P05067"]
})
from_db = "UniProtKB_AC-ID"
to_db = "UniProtKB"

### Instantiating the API

In [23]:
instance = UniprotInterface(
    total_retries=5
)

### Making the request

In [27]:
response = instance.download_batch(
    dataset=df,
    column_ids="ids",
    auto_db=False,
    from_db=from_db,
    to_db=to_db,
    batch_size=5
)

Processing manual IDs: 100%|██████████ 1/1 [00:03<00:00,  3.78s/it] Processing manual IDs

Fetched: 1 / 1


### Parsing results

Wether if we dont want to filter the results and get all the available fields, we can set `extract_fields=None`.

In [34]:
instance.parse_results(response, extract_fields=None)

,accession,protein_name,organism_name,gene_primary,taxon_id,ineage,sequence,length,alphafold_ids,biogrid_ids,...,pfam_ids,pride_ids,reactome_ids,refseq_ids,rhea_ids,string_ids,references,features,keywords,source_db
0,P05067,Amyloid-beta precursor protein,Homo sapiens,[APP],9606,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMN...,770,[P05067],[106848],...,"[PF10515, PF12924, PF12925, PF02177, PF03494, ...",[],"[R-HSA-114608, R-HSA-3000178, R-HSA-381426, R-...","[NP_000475.1, NP_001129488.1, NP_001129601.1, ...",[],[9606.ENSP00000284981],[{'title': 'The precursor of Alzheimer's disea...,"[{'type': 'Signal', 'description': '', 'locati...","[3D-structure, Alternative splicing, Alzheimer...",unknown


If we want to filter the results and get specific fields, we can set `extract_fields` to a list of desired fields. For example, to get the accession number, ID, protein name, gene names, organism name, and length, we can set `extract_fields` as follows:

In [29]:
instance.parse_results(response, extract_fields=["accession", "id", "protein_name", "gene_names", "organism_name", "length"])

,accession,protein_name,organism_name,length,source_db
0,P05067,Amyloid-beta precursor protein,Homo sapiens,770,unknown


## Sequence Search
For this search type it will need a list of sequences to be searched.

In [ ]:
## Pending sequence search example